In [1]:
import sys
from pathlib import Path

import pandas as pd

# Notebook chạy từ notebooks/, nên đưa gốc repo (chứa src/) vào sys.path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import METRICS_DIR
from src.golub_wv import GolubWeightedVoting, loocv
from src.load import load_golub

X, s = load_golub("curated")
tr = s["split"] == "train"
X_train, y_train = X[tr], s.loc[tr, "class"]
X_test, y_test = X[~tr], s.loc[~tr, "class"]
print("train:", X_train.shape, "| test:", X_test.shape)

train: (38, 7129) | test: (34, 7129)


In [2]:
cv = loocv(X_train, y_train, n_genes=50)
cv.head()

,true,pred,ps
sample_id,,,
1,ALL,ALL,0.961068
2,ALL,ALL,0.420353
3,ALL,ALL,0.849173
4,ALL,ALL,0.913442
5,ALL,ALL,0.898571


In [3]:
clf = GolubWeightedVoting(n_genes=50).fit(X_train, y_train)
test = pd.DataFrame({"true": y_test,
                     "pred": clf.predict(X_test),
                     "ps": clf.prediction_strength(X_test)}, index=X_test.index)
test.head()

,true,pred,ps
sample_id,,,
39,ALL,ALL,0.765209
40,ALL,ALL,0.538456
41,ALL,ALL,0.929406
42,ALL,ALL,0.459299
43,ALL,ALL,0.646143


In [4]:
def summarize(df):
    """Đếm theo cách bài báo báo cáo: số dự đoán chắc chắn, số đúng trong đó, số uncertain, PS trung vị."""
    sure = df["pred"] != "uncertain"
    return {"n": len(df),
            "chắc chắn": int(sure.sum()),
            "đúng": int((df.loc[sure, "pred"] == df.loc[sure, "true"]).sum()),
            "uncertain": int((~sure).sum()),
            "PS trung vị": round(float(df["ps"].median()), 2)}

paper_cv   = {"n": 38, "chắc chắn": 36, "đúng": 36, "uncertain": 2, "PS trung vị": 0.77}
paper_test = {"n": 34, "chắc chắn": 29, "đúng": 29, "uncertain": 5, "PS trung vị": 0.73}

compare = pd.DataFrame([
    {"tập": "LOOCV (38)",    "nguồn": "bài báo", **paper_cv},
    {"tập": "LOOCV (38)",    "nguồn": "nhóm",    **summarize(cv)},
    {"tập": "Kiểm tra (34)", "nguồn": "bài báo", **paper_test},
    {"tập": "Kiểm tra (34)", "nguồn": "nhóm",    **summarize(test)},
])
compare

,tập,nguồn,n,chắc chắn,đúng,uncertain,PS trung vị
0,LOOCV (38),bài báo,38,36,36,2,0.77
1,LOOCV (38),nhóm,38,36,36,2,0.75
2,Kiểm tra (34),bài báo,34,29,29,5,0.73
3,Kiểm tra (34),nhóm,34,33,32,1,0.73


In [5]:
both = pd.concat([cv.assign(set="loocv"), test.assign(set="test")])
both = both.join(s[["source", "tissue", "subtype"]])
both[both["pred"] != both["true"]]

,true,pred,ps,set,source,tissue,subtype
sample_id,,,,,,,
12,ALL,uncertain,0.036249,loocv,DFCI,BM,B-ALL
35,AML,uncertain,0.196143,loocv,CALGB,BM,AML
66,AML,ALL,0.300036,test,CCG,BM,AML
67,ALL,uncertain,0.122756,test,DFCI,PB,T-ALL


In [6]:
both.to_csv(METRICS_DIR / "golub_ps.csv")
compare.to_csv(METRICS_DIR / "golub_reproduction.csv", index=False)
print("Đã lưu:", METRICS_DIR / "golub_ps.csv", "và", METRICS_DIR / "golub_reproduction.csv")

Đã lưu: /Users/buituanhiep/Documents/golub-project/results/metrics/golub_ps.csv và /Users/buituanhiep/Documents/golub-project/results/metrics/golub_reproduction.csv
